# LeWM Dataset Goal Planning

This notebook samples a start/goal pair from the TwoRoom dataset, loads a frozen LeWM checkpoint, optimizes action sequences with CEM, and executes them in the true `swm/TwoRoom-v1` environment.

It includes a single open-loop rollout and a real receding-horizon MPC loop where later observations come from `env.step(...)`.


In [ ]:
from pathlib import Path
import os

from IPython.display import Video, display
import numpy as np
import stable_worldmodel as swm
from stable_worldmodel.data import HDF5Dataset
import torch

from constrained_jepa.data import fit_action_stats, sample_start_goal
from constrained_jepa.envs.tworoom import (
    make_tworoom_env,
    reset_tworoom_to_batch,
    rollout_action_blocks,
    run_env_mpc,
)
from constrained_jepa.planning.cem import CEMConfig, plan_cem
from constrained_jepa.visualization import (
    plot_env_trajectory,
    plot_mpc_costs,
    plot_plan,
    plot_start_goal,
    write_mp4,
)

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root containing pyproject.toml")


def format_num_params(num_params: int) -> str:
    if num_params >= 1_000_000:
        return f"{num_params / 1_000_000:.2f}M ({num_params:,})"
    if num_params >= 1_000:
        return f"{num_params / 1_000:.1f}K ({num_params:,})"
    return f"{num_params:,}"


def count_params(module: torch.nn.Module) -> int:
    return sum(param.numel() for param in module.parameters())


def print_jepa_param_counts(model: torch.nn.Module) -> None:
    part_names = ["encoder", "predictor", "action_encoder", "projector", "pred_proj"]
    total = count_params(model)
    covered = 0

    print("JEPA parameter counts:")
    for name in part_names:
        part = getattr(model, name, None)
        if part is None:
            continue
        num_params = count_params(part)
        covered += num_params
        print(f"  {name}: {format_num_params(num_params)}")

    if covered and covered != total:
        print(f"  other: {format_num_params(total - covered)}")
    print(f"  total: {format_num_params(total)}")

ROOT = find_repo_root()
STABLEWM_HOME = ROOT / "artifacts" / "stablewm"
os.environ["STABLEWM_HOME"] = str(STABLEWM_HOME)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
TWO_ROOM_SUCCESS_THRESHOLD = 1.0

print(f"Using device: {DEVICE}")
print(f"Using seed: {SEED}")
print(f"TwoRoom success threshold: {TWO_ROOM_SUCCESS_THRESHOLD}")
print("STABLEWM_HOME:", STABLEWM_HOME)


## Load Model And Dataset

TwoRoom is the smallest extracted dataset here and a good first target for debugging the planning loop.


In [ ]:
# Dataset/model parameters.
DATASET_NAME = "tworoom"
MODEL_NAME = "tworoom/lewm"

# Structural environment/model dimensions.
ENV_ACTION_DIM = 2
ACTION_BLOCK = 5     # Related to the frameskip parameter used when the model was trained


In [ ]:
dataset = HDF5Dataset(DATASET_NAME, cache_dir=STABLEWM_HOME, keys_to_cache=["action"])
action_mean, action_std = fit_action_stats(dataset)

model = swm.policy.AutoCostModel(MODEL_NAME, cache_dir=STABLEWM_HOME)
model = model.to(DEVICE).eval()


In [ ]:
print("dataset samples:", len(dataset))
print("columns:", dataset.column_names)
print("action mean/std:", action_mean.tolist(), action_std.tolist())
print_jepa_param_counts(model)


## Sample Start And Goal


In [ ]:
# Start/goal sampling parameters.
START_GOAL_HISTORY = 3
GOAL_OFFSET = 60


In [ ]:
rng = np.random.default_rng(SEED)
batch = sample_start_goal(
    dataset,
    rng=rng,
    history=START_GOAL_HISTORY,
    env_action_dim=ENV_ACTION_DIM,
    action_block=ACTION_BLOCK,
    goal_offset=GOAL_OFFSET,
    action_mean=action_mean,
    action_std=action_std,
)


In [ ]:
print("episode:", batch.episode_id)
print("start step:", batch.start_step)
print("goal step:", batch.goal_step)
print({key: tuple(value.shape) for key, value in batch.info.items()})

plot_start_goal(batch.raw_start_pixels, batch.raw_goal_pixels);


## Open loop planning


In [ ]:
# Optimizer-intrinsic CEM parameters.
CEM_NUM_SAMPLES = 1024
CEM_ELITES_FRACTION = 0.1
CEM_NUM_ELITES = int(CEM_NUM_SAMPLES * CEM_ELITES_FRACTION)
CEM_NUM_ITERS = 20

# Planner parameters, expressed in raw environment action steps.
PLANNER_HORIZON = 50

# Open-loop env rollout/video parameters.
OPEN_LOOP_VIDEO_FPS = 10
OPEN_LOOP_VIDEO_PATH = ROOT / "artifacts" / "videos" / "tworoom_open_loop.mp4"


In [ ]:
open_loop_plan = plan_cem(
    model,
    batch.info,
    planner_horizon=PLANNER_HORIZON,
    env_action_dim=ENV_ACTION_DIM,
    action_block=ACTION_BLOCK,
    optimizer=CEMConfig(
        num_samples=CEM_NUM_SAMPLES,
        num_elites=CEM_NUM_ELITES,
        num_iters=CEM_NUM_ITERS,
    ),
    device=DEVICE,
)


In [ ]:
print("final best cost:", open_loop_plan.costs[-1])
print("best cost:", min(open_loop_plan.costs))
print("final average cost:", open_loop_plan.average_costs[-1])
plot_plan(open_loop_plan.actions, open_loop_plan.costs, open_loop_plan.average_costs);


In [ ]:
env = make_tworoom_env(render_target=True, success_threshold=TWO_ROOM_SUCCESS_THRESHOLD)
try:
    reset_tworoom_to_batch(env, dataset, batch)
    open_loop_rollout = rollout_action_blocks(
        env,
        open_loop_plan.actions,
        action_mean=action_mean,
        action_std=action_std,
        env_action_dim=ENV_ACTION_DIM,
        action_block=ACTION_BLOCK,
        num_steps=PLANNER_HORIZON,
    )
    open_loop_video_path = write_mp4(
        open_loop_rollout.frames,
        OPEN_LOOP_VIDEO_PATH,
        fps=OPEN_LOOP_VIDEO_FPS,
    )
finally:
    env.close()


In [ ]:
print("env rollout steps:", len(open_loop_rollout.actions))
print("success threshold:", TWO_ROOM_SUCCESS_THRESHOLD)
print("terminated:", open_loop_rollout.terminated)
print("truncated:", open_loop_rollout.truncated)
print("final distance:", open_loop_rollout.distances[-1] if open_loop_rollout.distances else None)
print("video:", open_loop_video_path)
display(
    Video(
        str(open_loop_video_path),
        embed=True,
        width=400,
    )
)


## Closed loop

Run receding-horizon MPC in the real TwoRoom environment. The planner replans every `REPLAN_FREQUENCY` raw env steps and executes actions through `env.step(...)`.


In [ ]:
# Closed-loop MPC parameters, expressed in raw environment action steps.
REPLAN_FREQUENCY = 1
TRAJECTORY_HORIZON = 50

# Closed-loop env rollout/video parameters.
CLOSED_LOOP_VIDEO_FPS = 10
CLOSED_LOOP_VIDEO_PATH = ROOT / "artifacts" / "videos" / "tworoom_closed_loop.mp4"


In [ ]:
env = make_tworoom_env(render_target=True, success_threshold=TWO_ROOM_SUCCESS_THRESHOLD)
try:
    closed_loop = run_env_mpc(
        model,
        env,
        dataset,
        batch,
        planner_horizon=PLANNER_HORIZON,
        replan_frequency=REPLAN_FREQUENCY,
        trajectory_horizon=TRAJECTORY_HORIZON,
        env_action_dim=ENV_ACTION_DIM,
        action_block=ACTION_BLOCK,
        action_mean=action_mean,
        action_std=action_std,
        optimizer=CEMConfig(
            num_samples=CEM_NUM_SAMPLES,
            num_elites=CEM_NUM_ELITES,
            num_iters=CEM_NUM_ITERS,
        ),
        device=DEVICE,
    )
    closed_loop_video_path = write_mp4(
        closed_loop.frames,
        CLOSED_LOOP_VIDEO_PATH,
        fps=CLOSED_LOOP_VIDEO_FPS,
    )
finally:
    env.close()


In [ ]:
for i, step in enumerate(closed_loop.steps):
    avg_costs = getattr(step, "average_costs", None)
    avg_text = f", avg_final={avg_costs[-1]:.3f}" if avg_costs else ""
    print(
        f"replan {i}: best={min(step.costs):.3f}, final={step.costs[-1]:.3f}"
        f"{avg_text}, distance_after={step.distance_after:.3f}"
    )

print("env rollout steps:", len(closed_loop.distances))
print("success threshold:", TWO_ROOM_SUCCESS_THRESHOLD)
print("terminated:", closed_loop.terminated)
print("truncated:", closed_loop.truncated)
print("final distance:", closed_loop.distances[-1] if closed_loop.distances else None)
print("video:", closed_loop_video_path)

closed_loop_average_costs = [
    getattr(step, "average_costs", None) for step in closed_loop.steps
]
if all(costs for costs in closed_loop_average_costs):
    plot_mpc_costs(
        [step.costs for step in closed_loop.steps],
        closed_loop_average_costs,
    );
else:
    plot_mpc_costs([step.costs for step in closed_loop.steps]);

plot_env_trajectory(closed_loop.frames, closed_loop.positions);
display(
    Video(
        str(closed_loop_video_path),
        embed=True,
        width=400,
    )
)
